## 2D PEPS

Number of variational parameters in a PEPS with bond dimension D on $L\times L$ square lattice:
$$N(A_{lrud}^\alpha) = D^4 \times 2 \times L^2$$

## Complexity

SVD:

For $m\times n$ matrix,
$$O(m^3+mn^2+n^3)$$

QR:

For $n\times n$ real square matrix,
$$O(n^3)$$

## Notes for variables in the codes and recap for natural gradient descent method

Symbols in the calculation of local energy and gradient.
$$cx = \langle S|\psi \rangle$$
$$exy = \langle S|H_{xy}|\psi\rangle$$
$$eu = E_z = \langle S | H_z | \psi \rangle$$
$$(vx)_i = \langle S | \partial_{\theta_i}|\psi\rangle/ \langle S|\psi\rangle = \langle S |\psi_i\rangle/ \langle S|\psi\rangle$$
$$\Rightarrow E_{loc} = \langle S|H|\psi\rangle/ \langle S|\psi\rangle = exy/cx+eu \triangleq ex$$
where we denote $\partial_{\theta_i}|\psi\rangle = |\psi_i\rangle$ for simplicity. 

Gradient terms used in quantum natural gradient descent (stochastic reconfiguration, SR):
$$g_i = \langle \psi_i|H|\psi\rangle / \langle\psi|\psi\rangle - \langle \psi|H|\psi\rangle \langle \psi_i|\psi \rangle/ \langle\psi|\psi\rangle^2 $$
$$S_{ij} = \langle\psi_i|\psi_j\rangle/\langle\psi|\psi\rangle - \langle\psi_i|\psi\rangle \langle\psi_j|\psi\rangle/\langle\psi|\psi\rangle^2$$
$$R = S+\eta I$$
$S$ is called the Quantum Geometric Tensor (QGT). In natural gradient descent, it is originally called the Fisher information metric, which defines the Riemannian space for the parameters in a probability statistical model. As a simple example, consider a discrete probability model, then since it represents probability distribution there's a natural normalization constraint on its output probabilities as $\sum_i p_i=1$. We want to describe the space where the model outputs live in (for the purpose of learning another distribution), in the language of quantum states, we want to study the variational state output in the corresponding projected Hilbert space. The requirement of 'probablistic'/'normalization' add constraints to the output of the statistical model, making an (sphere, as in discrete case) embedding into an Euclidean space and thus inducing a metric on the sphere (locally it is the same as Euclidean metric). The model $p(x|\theta)$ introduces a change of variables from $p$ to $\theta$. After such change of variables, the original Euclidean metric on the sphere becomes the Fisher information metric. From Wiki: __"That is, the Fisher information metric on a statistical manifold is simply (four times) the Euclidean metric restricted to the positive orthant of the sphere, after appropriate changes of variable."__

For complex Hilbert space, the concept of Fisher metric for real probabilistic model can be extended to Fubini–Study metric. __Equivalently, the Fubini–Study metric can be understood as the metric on complex projective Hilbert space that is induced by the complex extension of the flat Euclidean metric.__

SR serves as a preconditioner on the gradient used in stochastic optimization. In VMC setting, the above quantity can be computed as statistical average value of the local quantities under the empirical probability measure. E.g. the gradient:
$$g_i = \langle \psi_i|H|\psi\rangle / \langle\psi|\psi\rangle - \langle E\rangle \langle \psi_i/\psi\rangle = \frac{1}{Z} \sum_\sigma \langle \psi_i|\sigma\rangle \langle \sigma|H|\psi\rangle  - \langle E\rangle \langle \psi_i/\psi\rangle = \frac{1}{Z} \sum_\sigma \langle \sigma|\psi\rangle^2 \frac{\psi_i(\sigma)}{\psi(\sigma)} E_{loc}(\sigma)  - \langle E\rangle \langle \psi_i/\psi\rangle,$$
and for the SR QGT:
$$S_{ij} = \langle \frac{\psi_i(\sigma)}{\psi(\sigma)}\times\frac{\psi_j(\sigma)}{\psi(\sigma)}\rangle - \langle \frac{\psi_i(\sigma)}{\psi(\sigma)}\rangle \langle\frac{\psi_j(\sigma)}{\psi(\sigma)}\rangle$$

In natural gradient descent method, we would like to minimize the energy difference (can be negative) between two optimization steps, which is 
$$\Delta \varepsilon = \varepsilon(\psi_{\theta+\delta})-\varepsilon(\psi_\theta) = \delta^*g+g^*\delta$$
we add a penalization term $\delta^*S\delta/\epsilon$ to keep the update step small. Therefore, the gradient descent is formalized as a minimization problem in terms of $\delta$:
$$\min_\delta [\delta^*g+g^*\delta + \delta^*R\delta/\epsilon]$$
here $R = S+\eta I$ to avoid singularity when taking its inverse. By Cauchy-Schwarz inequality we can show that QGT $S$ is positive semi-definite and R is thus positive definite. Actually it can be shown that by adding this penalty term we can minimize the distance between the updated state $|\psi_{\theta + \delta}\rangle$ and the one-step imaginary-time updated state $e^{-\epsilon H}|\psi_{\theta}\rangle$.

By solving the minimization problem we obtain that the step $\delta$ should be set as:
$$\delta = -\epsilon R^{-1}g.$$

A complication is given by the fact that the QGT is determined by Monte Carlo sampling and it might have several eigenvalues that are zero or very small, leading to numerical stability issues when inverting the matrix or in the resulting dynamics.

### Debug notes

1. For cubic case, especially 4x4x4 system, the sampling update rule is crucial for numerical stability. If we use update rule that conserves the total spin Sz (NNASP), it is likely that we sample some weird configurations that have low probability and thus may give rise to numerical singularity during gradient backpropagation calculation (like degenerate singular values). Thus to address this we choose another update rule (flipping the spins on each site one by one) which does not converve the total spin.

(!!) This change helps reduce the gradient explosion but still cannot completely circumvent it.

In [ ]:
# ENERGY GRADIENT CALCULATION ANALYSIS

Energy as an expectation:
  $$E(\theta) = \sum_x p_\theta(x), E_{\text{loc}}(x;\theta), \quad p_\theta(x) = \frac{|\psi_\theta(x)|^2}{Z}, \quad E_{\text{loc}}(x) = \sum_{x'} H_{xx'}\frac{\psi(x')}{\psi(x)}$$

  Define the log-derivative: $O(x) = \partial_\theta \log\psi(x)$.
  
Method 1: Analytical grad, then MC estimate

  Differentiate $E = \sum_x p(x) E_{\text{loc}}(x)$ w.r.t. $\theta$:

  $$\partial_\theta E = \underbrace{\sum_x (\partial_\theta p(x)), E_{\text{loc}}(x)}_{\text{(A): distribution shift}} + \underbrace{\sum_x p(x), \partial\theta E_{\text{loc}}(x)}_{\text{(B): local energy change}}$$

  Term (A): Since $\partial_\theta \log p(x) = 2O(x) - 2\langle O\rangle$ (for real $\psi$):

  $$(\text{A}) = 2\sum_x p(x)\bigl[O(x) - \langle O\rangle\bigr] E_{\text{loc}}(x) = 2,\text{Cov}p(O,; E{\text{loc}})$$

  Term (B): Compute $\partial_\theta E_{\text{loc}}(x)$:

  $$\partial_\theta E_{\text{loc}}(x) = \sum_{x'} H_{xx'}\frac{\psi(x')}{\psi(x)}\bigl[O(x') - O(x)\bigr]$$

  Now take the expectation:

  $$(\text{B}) = \frac{1}{Z}\sum_{x,x'} \psi(x), H_{xx'}, \psi(x')\bigl[O(x') - O(x)\bigr]$$

  Swap $x \leftrightarrow x'$ in the $O(x')$ sum and use $H_{x'x} = H_{xx'}$ (Hermiticity, real $H$):

  $$\sum_{x,x'}\psi(x)H_{xx'}\psi(x')O(x') = \sum_{x,x'}\psi(x')H_{xx'}\psi(x)O(x) = \sum_{x,x'}\psi(x)H_{xx'}\psi(x')O(x)$$

  The two sums are identical, so (B) = 0 exactly for Hermitian $H$ and real $\psi$.

  Result:
  $$\boxed{\partial_\theta E = 2,\text{Cov}p\bigl(O,; E{\text{loc}}\bigr)}$$

  MC estimator: sample ${x_i} \sim p_\theta$, compute $\frac{2}{N_s}\sum_i \bigl(O(x_i) - \bar{O}\bigr)\bigl(E_{\text{loc}}(x_i) - \bar{E}_{\text{loc}}\bigr)$.

  ---
  Method 2: MC estimate of E, then differentiate

  Fix samples ${x_i}$ drawn from $p_\theta$, form:

  $$\hat{E}(\theta) = \frac{1}{N_s}\sum_i E_{\text{loc}}(x_i;,\theta)$$

  Then differentiate w.r.t. $\theta$ at fixed ${x_i}$:

  $$\nabla_\theta \hat{E} = \frac{1}{N_s}\sum_i \nabla_\theta E_{\text{loc}}(x_i;,\theta) = \frac{1}{N_s}\sum_i \sum_{x'} H_{x_i x'}\frac{\psi(x')}{\psi(x_i)}\bigl[O(x') - O(x_i)\bigr]$$

  This is a MC estimator of term (B) alone, which we just showed equals zero in expectation.

  Result:
  $$\boxed{\mathbb{E}{p\theta}\bigl[\nabla_\theta E_{\text{loc}}\bigr] = 0}$$

  So Method 2 estimates zero, not the energy gradient.

  ---
  Why they differ — intuition

  When you change $\theta$, the energy changes for two reasons:
  1. Distribution shift — different configurations get different weights $p_\theta(x)$
  2. Local energy change — $E_{\text{loc}}(x;\theta)$ itself changes at each $x$

  For Hermitian $H$ (real $\psi$), reason (2) cancels exactly by the swap symmetry above. All of the gradient comes from the distribution shift (reason 1).

  Method 2 only captures reason (2) — it misses the REINFORCE/score-function term entirely. This is why VMC uses the $2,\text{Cov}(O, E_{\text{loc}})$ formula: it's the score-function estimator that captures the distribution shift.

  ---
  Important caveat: complex $\psi$

  For complex $\psi$, the swap trick gives $\sum \psi^(x)H_{xx'}\psi(x')O(x')$ vs $\sum \psi^(x)H_{xx'}\psi(x')O(x)$. After swapping, the first becomes $\sum \psi^(x')H^_{xx'}\psi(x)O(x)$, which is the complex conjugate of the second only if
  $O$ is real. So term (B) = 0 still holds for real parameters, but needs more care for complex parameters.

  In your codebase (real fPEPS amplitudes, real parameters), the cancellation is exact.